In [ ]:
# ============================================================
# SDP II: Sparse Portfolio QUBO Benchmarking with QAOA
# Solvers:
#   1. CPLEX / brute force exact reference
#   2. Simulated Annealing
#   3. Simulated Quantum Annealing
#   4. QAOA statevector simulation
#   5. QAOA shot-based simulation
#   6. Optional IBM hardware execution
#
# Author: Abdulrahman Alnuaimi
# ============================================================

# ============================================================
# Install commands if needed:
# pip install numpy pandas scipy dimod dwave-samplers docplex cplex qiskit qiskit-aer qiskit-ibm-runtime
# ============================================================

import time
import json
import itertools
import warnings

import numpy as np
import pandas as pd
import dimod

from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector

warnings.filterwarnings("ignore")


# ============================================================
# Optional imports
# ============================================================

try:
    from qiskit_aer import AerSimulator
    HAS_AER = True
except Exception:
    HAS_AER = False

try:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as IBMSampler
    HAS_IBM_RUNTIME = True
except Exception:
    HAS_IBM_RUNTIME = False

try:
    from dwave.samplers import SimulatedAnnealingSampler, PathIntegralAnnealingSampler
    HAS_DWAVE_SAMPLERS = True
except Exception:
    HAS_DWAVE_SAMPLERS = False

try:
    from docplex.mp.model import Model
    HAS_CPLEX = True
except Exception:
    HAS_CPLEX = False


# ============================================================
# Experiment settings
# ============================================================

ASSET_LIST = [4, 6, 8, 10, 12]
DENSITY_LIST = [0.25]
SEED_LIST = [1, 2, 3]

BUDGET_MODE = "half"

RISK_AVERSION = 1.0
PENALTY = 20.0

SA_NUM_READS = 200
SA_SWEEPS = 1000

SQA_NUM_READS = 200
SQA_SWEEPS = 1000

QAOA_P_LIST = [1, 2, 3]
QAOA_MAXITER = 60
QAOA_SHOTS = 1024

RUN_CPLEX = True
RUN_SA = True
RUN_SQA = True
RUN_QAOA_STATEVECTOR = True
RUN_QAOA_SHOTS = True

# Set to True only when you want to run real IBM hardware.
RUN_IBM_HARDWARE = False

IBM_BACKEND_NAME = None
IBM_SHOTS = 1024

OUTPUT_RESULTS_CSV = "sdp2_sparse_portfolio_results.csv"
OUTPUT_SUMMARY_CSV = "sdp2_sparse_portfolio_summary.csv"


# ============================================================
# Basic helpers
# ============================================================

def budget_from_mode(n, mode="half"):
    if mode == "half":
        return max(1, n // 2)
    if mode == "third":
        return max(1, n // 3)
    if mode == "fixed_2":
        return min(2, n)
    raise ValueError(f"Unknown budget mode: {mode}")


def selected_count(bitstring):
    return int(sum(int(c) for c in bitstring))


def compute_gap(energy, exact_energy):
    if exact_energy is None:
        return None

    return float((energy - exact_energy) / max(abs(exact_energy), 1e-12))


def qiskit_label_to_portfolio_bitstring(label, n):
    clean = label.replace(" ", "")
    clean = clean.zfill(n)

    # Qiskit bitstrings are little-endian relative to our x0...xn convention.
    return clean[::-1]


# ============================================================
# Sparse portfolio QUBO generation
# ============================================================

def generate_sparse_covariance(n, density=0.25, seed=1):
    rng = np.random.default_rng(seed)

    mu = rng.normal(loc=0.10, scale=0.05, size=n)

    A = rng.normal(size=(n, n))
    cov = A @ A.T
    cov = cov / np.max(np.abs(cov))

    if density < 1.0:
        mask = np.zeros((n, n), dtype=bool)

        for i in range(n):
            mask[i, i] = True

        for i in range(n):
            for j in range(i + 1, n):
                keep = rng.random() < density
                mask[i, j] = keep
                mask[j, i] = keep

        cov = cov * mask

        # Diagonal dominance makes the sparse covariance more stable.
        for i in range(n):
            off_diag_sum = np.sum(np.abs(cov[i])) - abs(cov[i, i])
            cov[i, i] = max(cov[i, i], off_diag_sum + 0.05)

    return mu, cov


def build_portfolio_bqm(mu, cov, budget, risk_aversion=1.0, penalty=20.0):
    n = len(mu)

    linear = {}
    quadratic = {}
    offset = penalty * (budget ** 2)

    # Return and diagonal risk terms
    for i in range(n):
        linear[i] = -mu[i] + risk_aversion * cov[i, i]

    # Off-diagonal risk terms:
    # x^T Sigma x = sum_i Sigma_ii x_i + 2 sum_i<j Sigma_ij x_i x_j
    for i in range(n):
        for j in range(i + 1, n):
            if abs(cov[i, j]) > 1e-12:
                quadratic[(i, j)] = quadratic.get((i, j), 0.0) + 2.0 * risk_aversion * cov[i, j]

    # Penalty expansion:
    # A(sum x_i - k)^2
    for i in range(n):
        linear[i] += penalty * (1 - 2 * budget)

    for i in range(n):
        for j in range(i + 1, n):
            quadratic[(i, j)] = quadratic.get((i, j), 0.0) + 2.0 * penalty

    return dimod.BinaryQuadraticModel(linear, quadratic, offset, dimod.BINARY)


def energy_of_bitstring(bqm, bitstring):
    sample = {i: int(bitstring[i]) for i in range(len(bitstring))}
    return float(bqm.energy(sample))


def sample_to_bitstring(sample, n):
    return "".join(str(int(sample[i])) for i in range(n))


# ============================================================
# Exact solvers
# ============================================================

def brute_force_exact(bqm, n):
    best_energy = float("inf")
    best_bitstring = None

    for bits in itertools.product([0, 1], repeat=n):
        bitstring = "".join(str(b) for b in bits)
        energy = energy_of_bitstring(bqm, bitstring)

        if energy < best_energy:
            best_energy = energy
            best_bitstring = bitstring

    return best_bitstring, best_energy


def solve_cplex_exact(bqm, n):
    if not HAS_CPLEX:
        return None, None, "CPLEX not installed"

    mdl = Model(name="portfolio_qubo")

    x = {i: mdl.binary_var(name=f"x_{i}") for i in range(n)}

    obj = bqm.offset

    for i, bias in bqm.linear.items():
        obj += bias * x[i]

    for (i, j), bias in bqm.quadratic.items():
        obj += bias * x[i] * x[j]

    mdl.minimize(obj)

    sol = mdl.solve(log_output=False)

    if sol is None:
        return None, None, "CPLEX failed"

    bitstring = "".join(str(int(round(sol[x[i]]))) for i in range(n))
    energy = energy_of_bitstring(bqm, bitstring)

    return bitstring, energy, "CPLEX exact"


# ============================================================
# SA and SQA
# ============================================================

def solve_dwave_sampler(bqm, n, sampler, num_reads=200, sweeps=1000):
    start = time.perf_counter()

    try:
        sampleset = sampler.sample(
            bqm,
            num_reads=num_reads,
            num_sweeps=sweeps
        )
    except TypeError:
        try:
            sampleset = sampler.sample(
                bqm,
                num_reads=num_reads,
                sweeps=sweeps
            )
        except TypeError:
            sampleset = sampler.sample(
                bqm,
                num_reads=num_reads
            )

    wall_time = time.perf_counter() - start

    best = sampleset.first
    best_bitstring = sample_to_bitstring(best.sample, n)
    best_energy = float(best.energy)

    total_reads = 0
    best_reads = 0

    for row in sampleset.data(["energy", "num_occurrences"]):
        total_reads += row.num_occurrences

        if abs(float(row.energy) - best_energy) < 1e-9:
            best_reads += row.num_occurrences

    success_prob = best_reads / total_reads if total_reads > 0 else None

    return best_bitstring, best_energy, success_prob, wall_time


# ============================================================
# QAOA helpers
# ============================================================

def bqm_binary_to_spin_coefficients(bqm):
    spin_bqm = bqm.change_vartype(dimod.SPIN, inplace=False)

    h = dict(spin_bqm.linear)
    J = dict(spin_bqm.quadratic)
    offset = float(spin_bqm.offset)

    return h, J, offset


def build_qaoa_circuit_rx(bqm, n, gammas, betas, measure=False):
    h, J, offset = bqm_binary_to_spin_coefficients(bqm)

    qc = QuantumCircuit(n)

    for q in range(n):
        qc.h(q)

    p = len(gammas)

    for layer in range(p):
        gamma = gammas[layer]
        beta = betas[layer]

        for i, coeff in h.items():
            qc.rz(2.0 * gamma * coeff, i)

        for (i, j), coeff in J.items():
            qc.rzz(2.0 * gamma * coeff, i, j)

        for q in range(n):
            qc.rx(2.0 * beta, q)

    if measure:
        qc.measure_all()

    return qc


def statevector_probabilities_for_portfolio(qc_no_measure, n):
    sv = Statevector.from_instruction(qc_no_measure)
    raw_probs = sv.probabilities_dict()

    probs = {}

    for label, prob in raw_probs.items():
        bitstring = qiskit_label_to_portfolio_bitstring(label, n)
        probs[bitstring] = probs.get(bitstring, 0.0) + float(prob)

    return probs


def expected_energy_from_probs(bqm, probs):
    total = 0.0

    for bitstring, prob in probs.items():
        total += prob * energy_of_bitstring(bqm, bitstring)

    return float(total)


def best_bitstring_from_probs(bqm, probs):
    best_bitstring = None
    best_energy = float("inf")
    best_prob = None

    for bitstring, prob in probs.items():
        energy = energy_of_bitstring(bqm, bitstring)

        if energy < best_energy:
            best_energy = energy
            best_bitstring = bitstring
            best_prob = prob

    return best_bitstring, best_energy, best_prob


def optimize_qaoa_statevector(bqm, n, p, maxiter=60, seed=1):
    rng = np.random.default_rng(seed)

    init_gammas = rng.uniform(0.0, np.pi, size=p)
    init_betas = rng.uniform(0.0, np.pi / 2, size=p)

    x0 = np.concatenate([init_gammas, init_betas])

    def objective(params):
        gammas = params[:p]
        betas = params[p:]

        qc = build_qaoa_circuit_rx(
            bqm=bqm,
            n=n,
            gammas=gammas,
            betas=betas,
            measure=False
        )

        probs = statevector_probabilities_for_portfolio(qc, n)
        return expected_energy_from_probs(bqm, probs)

    start = time.perf_counter()

    opt = minimize(
        objective,
        x0,
        method="COBYLA",
        options={
            "maxiter": maxiter,
            "rhobeg": 0.5,
            "tol": 1e-3,
            "disp": False
        }
    )

    wall_time = time.perf_counter() - start

    gammas = opt.x[:p]
    betas = opt.x[p:]

    final_qc = build_qaoa_circuit_rx(
        bqm=bqm,
        n=n,
        gammas=gammas,
        betas=betas,
        measure=False
    )

    probs = statevector_probabilities_for_portfolio(final_qc, n)

    best_bitstring, best_energy, best_prob = best_bitstring_from_probs(bqm, probs)

    return {
        "gammas": gammas,
        "betas": betas,
        "best_bitstring": best_bitstring,
        "best_energy": best_energy,
        "success_prob": best_prob,
        "wall_time": wall_time,
        "optimizer_fun": float(opt.fun),
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
        "probs": probs
    }


def run_qaoa_shot_simulator(bqm, n, gammas, betas, shots=1024):
    if not HAS_AER:
        return None, None, None, None, "qiskit-aer not installed"

    qc = build_qaoa_circuit_rx(
        bqm=bqm,
        n=n,
        gammas=gammas,
        betas=betas,
        measure=True
    )

    backend = AerSimulator()

    start = time.perf_counter()

    tqc = transpile(qc, backend)
    result = backend.run(tqc, shots=shots).result()

    wall_time = time.perf_counter() - start

    counts = result.get_counts()

    probs = {}

    for label, count in counts.items():
        bitstring = qiskit_label_to_portfolio_bitstring(label, n)
        probs[bitstring] = probs.get(bitstring, 0.0) + count / shots

    best_bitstring, best_energy, best_prob = best_bitstring_from_probs(bqm, probs)

    return best_bitstring, best_energy, best_prob, wall_time, "QAOA shot simulator"


# ============================================================
# IBM hardware
# ============================================================

def extract_sampler_counts(result):
    try:
        return result[0].data.meas.get_counts()
    except Exception:
        pass

    try:
        return result[0].data.c.get_counts()
    except Exception:
        pass

    raise RuntimeError("Could not extract counts from IBM Sampler result.")


def run_qaoa_ibm_hardware(bqm, n, gammas, betas, shots=1024, backend_name=None):
    if not HAS_IBM_RUNTIME:
        return None, None, None, None, None, None, "qiskit-ibm-runtime not installed"

    service = QiskitRuntimeService()

    if backend_name is None:
        backend = service.least_busy(
            operational=True,
            simulator=False,
            min_num_qubits=n
        )
    else:
        backend = service.backend(backend_name)

    qc = build_qaoa_circuit_rx(
        bqm=bqm,
        n=n,
        gammas=gammas,
        betas=betas,
        measure=True
    )

    start_transpile = time.perf_counter()

    tqc = transpile(
        qc,
        backend=backend,
        optimization_level=1
    )

    transpile_time = time.perf_counter() - start_transpile

    sampler = IBMSampler(mode=backend)

    start_run = time.perf_counter()

    job = sampler.run([tqc], shots=shots)
    result = job.result()

    wall_time = time.perf_counter() - start_run

    counts = extract_sampler_counts(result)

    probs = {}

    for label, count in counts.items():
        bitstring = qiskit_label_to_portfolio_bitstring(label, n)
        probs[bitstring] = probs.get(bitstring, 0.0) + count / shots

    best_bitstring, best_energy, best_prob = best_bitstring_from_probs(bqm, probs)

    qpu_seconds = None

    try:
        usage = job.usage_estimation()
        qpu_seconds = usage.get("quantum_seconds", None)
    except Exception:
        pass

    note = f"backend={backend.name}; job_id={job.job_id()}; depth={tqc.depth()}"

    return best_bitstring, best_energy, best_prob, wall_time, transpile_time, qpu_seconds, note


# ============================================================
# Result helpers
# ============================================================

def make_result_row(
    instance_id,
    n,
    budget,
    density,
    seed,
    solver,
    energy,
    exact_energy,
    best_bitstring,
    success_prob,
    wall_time,
    p=None,
    shots=None,
    sweeps=None,
    num_reads=None,
    maxiter=None,
    qpu_seconds=None,
    transpile_time=None,
    note=""
):
    return {
        "instance_id": instance_id,
        "n": n,
        "budget": budget,
        "density": density,
        "seed": seed,
        "solver": solver,
        "p": p,
        "shots": shots,
        "sweeps": sweeps,
        "num_reads": num_reads,
        "maxiter": maxiter,
        "best_bitstring": best_bitstring,
        "selected": selected_count(best_bitstring) if best_bitstring is not None else None,
        "energy": energy,
        "exact_energy": exact_energy,
        "gap": compute_gap(energy, exact_energy) if energy is not None else None,
        "success_prob": success_prob,
        "wall_time_sec": wall_time,
        "transpile_time_sec": transpile_time,
        "qpu_seconds": qpu_seconds,
        "note": note
    }


# ============================================================
# Main benchmark
# ============================================================

def run_full_benchmark():
    results = []

    qaoa_params = {}

    for n in ASSET_LIST:
        for density in DENSITY_LIST:
            for seed in SEED_LIST:

                budget = budget_from_mode(n, BUDGET_MODE)
                instance_id = f"n{n}_k{budget}_d{density}_seed{seed}"

                print("\n=================================================")
                print("Instance:", instance_id)
                print("=================================================")

                mu, cov = generate_sparse_covariance(
                    n=n,
                    density=density,
                    seed=seed
                )

                bqm = build_portfolio_bqm(
                    mu=mu,
                    cov=cov,
                    budget=budget,
                    risk_aversion=RISK_AVERSION,
                    penalty=PENALTY
                )

                exact_bitstring = None
                exact_energy = None

                # ----------------------------
                # Exact reference
                # ----------------------------
                if RUN_CPLEX:
                    print("Running exact reference...")

                    start = time.perf_counter()

                    bitstring, energy, note = solve_cplex_exact(bqm, n)

                    if bitstring is None:
                        if n <= 20:
                            bitstring, energy = brute_force_exact(bqm, n)
                            note = "brute force exact"
                        else:
                            note = "exact skipped"

                    wall_time = time.perf_counter() - start

                    exact_bitstring = bitstring
                    exact_energy = energy

                    if bitstring is not None:
                        results.append(
                            make_result_row(
                                instance_id=instance_id,
                                n=n,
                                budget=budget,
                                density=density,
                                seed=seed,
                                solver="CPLEX_OR_EXACT",
                                energy=energy,
                                exact_energy=energy,
                                best_bitstring=bitstring,
                                success_prob=1.0,
                                wall_time=wall_time,
                                note=note
                            )
                        )

                        print("Exact:", bitstring, "energy:", energy)

                # ----------------------------
                # SA
                # ----------------------------
                if RUN_SA and HAS_DWAVE_SAMPLERS:
                    print("Running SA...")

                    sampler = SimulatedAnnealingSampler()

                    bitstring, energy, success_prob, wall_time = solve_dwave_sampler(
                        bqm=bqm,
                        n=n,
                        sampler=sampler,
                        num_reads=SA_NUM_READS,
                        sweeps=SA_SWEEPS
                    )

                    results.append(
                        make_result_row(
                            instance_id=instance_id,
                            n=n,
                            budget=budget,
                            density=density,
                            seed=seed,
                            solver="SA",
                            energy=energy,
                            exact_energy=exact_energy,
                            best_bitstring=bitstring,
                            success_prob=success_prob,
                            wall_time=wall_time,
                            sweeps=SA_SWEEPS,
                            num_reads=SA_NUM_READS,
                            note="simulated annealing"
                        )
                    )

                    print("SA:", bitstring, "energy:", energy)

                # ----------------------------
                # SQA
                # ----------------------------
                if RUN_SQA and HAS_DWAVE_SAMPLERS:
                    print("Running SQA...")

                    sampler = PathIntegralAnnealingSampler()

                    bitstring, energy, success_prob, wall_time = solve_dwave_sampler(
                        bqm=bqm,
                        n=n,
                        sampler=sampler,
                        num_reads=SQA_NUM_READS,
                        sweeps=SQA_SWEEPS
                    )

                    results.append(
                        make_result_row(
                            instance_id=instance_id,
                            n=n,
                            budget=budget,
                            density=density,
                            seed=seed,
                            solver="SQA",
                            energy=energy,
                            exact_energy=exact_energy,
                            best_bitstring=bitstring,
                            success_prob=success_prob,
                            wall_time=wall_time,
                            sweeps=SQA_SWEEPS,
                            num_reads=SQA_NUM_READS,
                            note="simulated quantum annealing"
                        )
                    )

                    print("SQA:", bitstring, "energy:", energy)

                # ----------------------------
                # QAOA statevector
                # ----------------------------
                if RUN_QAOA_STATEVECTOR:
                    for p in QAOA_P_LIST:
                        print(f"Running QAOA statevector, p={p}...")

                        qaoa_result = optimize_qaoa_statevector(
                            bqm=bqm,
                            n=n,
                            p=p,
                            maxiter=QAOA_MAXITER,
                            seed=seed
                        )

                        qaoa_params[(instance_id, p)] = {
                            "gammas": qaoa_result["gammas"],
                            "betas": qaoa_result["betas"],
                            "bqm": bqm,
                            "n": n,
                            "budget": budget,
                            "density": density,
                            "seed": seed,
                            "exact_energy": exact_energy
                        }

                        results.append(
                            make_result_row(
                                instance_id=instance_id,
                                n=n,
                                budget=budget,
                                density=density,
                                seed=seed,
                                solver="QAOA_STATEVECTOR",
                                energy=qaoa_result["best_energy"],
                                exact_energy=exact_energy,
                                best_bitstring=qaoa_result["best_bitstring"],
                                success_prob=qaoa_result["success_prob"],
                                wall_time=qaoa_result["wall_time"],
                                p=p,
                                maxiter=QAOA_MAXITER,
                                shots="none",
                                note=f"optimizer_fun={qaoa_result['optimizer_fun']}"
                            )
                        )

                        print(
                            "QAOA SV:",
                            qaoa_result["best_bitstring"],
                            "energy:",
                            qaoa_result["best_energy"],
                            "prob:",
                            qaoa_result["success_prob"]
                        )

                # ----------------------------
                # QAOA shots
                # ----------------------------
                if RUN_QAOA_SHOTS:
                    for p in QAOA_P_LIST:
                        key = (instance_id, p)

                        if key not in qaoa_params:
                            continue

                        params = qaoa_params[key]

                        print(f"Running QAOA shots, p={p}...")

                        bitstring, energy, success_prob, wall_time, note = run_qaoa_shot_simulator(
                            bqm=bqm,
                            n=n,
                            gammas=params["gammas"],
                            betas=params["betas"],
                            shots=QAOA_SHOTS
                        )

                        results.append(
                            make_result_row(
                                instance_id=instance_id,
                                n=n,
                                budget=budget,
                                density=density,
                                seed=seed,
                                solver="QAOA_SHOTS",
                                energy=energy,
                                exact_energy=exact_energy,
                                best_bitstring=bitstring,
                                success_prob=success_prob,
                                wall_time=wall_time,
                                p=p,
                                shots=QAOA_SHOTS,
                                maxiter=QAOA_MAXITER,
                                note=note
                            )
                        )

                        print("QAOA shots:", bitstring, "energy:", energy, "prob:", success_prob)

                # ----------------------------
                # IBM hardware
                # ----------------------------
                if RUN_IBM_HARDWARE:
                    for p in QAOA_P_LIST:
                        key = (instance_id, p)

                        if key not in qaoa_params:
                            continue

                        params = qaoa_params[key]

                        # Keep hardware controlled.
                        # Change/remove these filters if needed.
                        if seed != 1:
                            continue

                        if p != 1:
                            continue

                        print(f"Running IBM hardware, p={p}...")

                        bitstring, energy, success_prob, wall_time, transpile_time, qpu_seconds, note = run_qaoa_ibm_hardware(
                            bqm=bqm,
                            n=n,
                            gammas=params["gammas"],
                            betas=params["betas"],
                            shots=IBM_SHOTS,
                            backend_name=IBM_BACKEND_NAME
                        )

                        results.append(
                            make_result_row(
                                instance_id=instance_id,
                                n=n,
                                budget=budget,
                                density=density,
                                seed=seed,
                                solver="QAOA_IBM_HARDWARE",
                                energy=energy,
                                exact_energy=exact_energy,
                                best_bitstring=bitstring,
                                success_prob=success_prob,
                                wall_time=wall_time,
                                p=p,
                                shots=IBM_SHOTS,
                                maxiter=QAOA_MAXITER,
                                qpu_seconds=qpu_seconds,
                                transpile_time=transpile_time,
                                note=note
                            )
                        )

                        print("IBM:", bitstring, "energy:", energy, "prob:", success_prob)

    df = pd.DataFrame(results)

    df.to_csv(OUTPUT_RESULTS_CSV, index=False)

    summary = (
        df.groupby(["solver", "n", "p"], dropna=False)
        .agg(
            mean_gap=("gap", "mean"),
            std_gap=("gap", "std"),
            mean_success_prob=("success_prob", "mean"),
            mean_wall_time_sec=("wall_time_sec", "mean"),
            mean_qpu_seconds=("qpu_seconds", "mean"),
            count=("gap", "count")
        )
        .reset_index()
    )

    summary.to_csv(OUTPUT_SUMMARY_CSV, index=False)

    print("\nSaved:", OUTPUT_RESULTS_CSV)
    print("Saved:", OUTPUT_SUMMARY_CSV)

    return df, summary


# ============================================================
# QAOA playground function
# ============================================================

def run_qaoa_playground(
    n=10,
    p=2,
    mode="sv",
    density=0.25,
    seed=1,
    penalty=20.0,
    risk=1.0,
    shots=1024,
    maxiter=60,
    top_k=10,
    backend_name=None
):
    budget = budget_from_mode(n, "half")

    mu, cov = generate_sparse_covariance(
        n=n,
        density=density,
        seed=seed
    )

    bqm = build_portfolio_bqm(
        mu=mu,
        cov=cov,
        budget=budget,
        risk_aversion=risk,
        penalty=penalty
    )

    if n <= 20:
        exact_bitstring, exact_energy = brute_force_exact(bqm, n)
    else:
        exact_bitstring, exact_energy = None, None

    print("\n===== INSTANCE =====")
    print("n:", n, "| p:", p, "| density:", density)
    print("budget:", budget, "| penalty:", penalty)
    print("exact:", exact_bitstring, "| energy:", exact_energy)

    print("\nOptimizing QAOA using statevector...")
    qaoa_result = optimize_qaoa_statevector(
        bqm=bqm,
        n=n,
        p=p,
        maxiter=maxiter,
        seed=seed
    )

    gammas = qaoa_result["gammas"]
    betas = qaoa_result["betas"]

    if mode == "sv":
        probs = qaoa_result["probs"]
        note = "statevector"

    elif mode == "shots":
        bitstring, energy, success_prob, wall_time, note = run_qaoa_shot_simulator(
            bqm=bqm,
            n=n,
            gammas=gammas,
            betas=betas,
            shots=shots
        )

        qc = build_qaoa_circuit_rx(bqm, n, gammas, betas, measure=True)
        backend = AerSimulator()
        tqc = transpile(qc, backend)
        result = backend.run(tqc, shots=shots).result()
        counts = result.get_counts()

        probs = {}
        for label, count in counts.items():
            bit = qiskit_label_to_portfolio_bitstring(label, n)
            probs[bit] = probs.get(bit, 0.0) + count / shots

        note = f"shot simulator, shots={shots}"

    elif mode == "hardware":
        bitstring, energy, success_prob, wall_time, transpile_time, qpu_seconds, note = run_qaoa_ibm_hardware(
            bqm=bqm,
            n=n,
            gammas=gammas,
            betas=betas,
            shots=shots,
            backend_name=backend_name
        )

        print("Hardware result:", bitstring, energy, success_prob, note)

        return None

    else:
        raise ValueError("mode must be 'sv', 'shots', or 'hardware'")

    rows = []

    for bitstring, prob in probs.items():
        energy = energy_of_bitstring(bqm, bitstring)

        rows.append({
            "bitstring": bitstring,
            "prob": prob,
            "energy": energy,
            "selected": selected_count(bitstring),
            "gap": compute_gap(energy, exact_energy)
        })

    df = pd.DataFrame(rows)

    print("\n===== MODE =====")
    print(note)

    print("\n===== BEST ENERGY =====")
    print(
        df.sort_values(
            by=["energy", "prob"],
            ascending=[True, False]
        ).head(top_k)
    )

    print("\n===== TOP BY PROBABILITY =====")
    print(
        df.sort_values(
            by=["prob", "energy"],
            ascending=[False, True]
        ).head(top_k)
    )

    print("\n===== TOP FEASIBLE BY PROBABILITY =====")
    feasible_df = df[df["selected"] == budget]
    print(
        feasible_df.sort_values(
            by=["prob", "energy"],
            ascending=[False, True]
        ).head(top_k)
    )

    print("\nTotal feasible probability:", feasible_df["prob"].sum())

    return df


# ============================================================
# Main execution
# ============================================================

if __name__ == "__main__":
    df_results, df_summary = run_full_benchmark()

    print("\n================ RESULT PREVIEW ================")
    print(df_results.head())

    print("\n================ SUMMARY PREVIEW ================")
    print(df_summary)